# Evaluation: rule-based fall detector on URFD

In `rule_based.ipynb` I tuned the State Machines detector's thresholds on Le2i.
Le2i was my validation set — the data I looked at while picking my thresholds.
(`V`, `N`, `P`, `angle_thresh`, `bbox_thresh`)

URFD is my held out test set. The rules below were never tuned
against it, so the metrics in this notebook are an honest measure of
how well the detector generalizes.

Two things actually differ between Le2i and URFD:

1. **Framerate.** Le2i = 25 fps. URFD = 30 fps. This matters for
   `hip_velocity` (per-second), and it's the only place I'll change
   anything in the pipeline.
2. **Camera & scene.** Different rooms, different actors, different
   lighting. The thresholds were not picked with any of this in mind.

Plan: keep every rule and threshold byte-identical to `rule_based.ipynb`,
swap in URFD's features, and read off the metrics. If the detector fails
in new ways here, those failures are the motivation for either going back 
and tuning my rule based approach/thresholds or moving to an LSTM in Phase 6.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


In [ ]:
# Load the per-frame URFD features extracted by phases/extract_features_urfd.py.
feature_df = pd.read_csv('../data/urfd_features.csv')

print('shape:', feature_df.shape)
print('columns:', feature_df.columns.tolist())
feature_df.head()


In [ ]:
# Same identifier convention as rule_based.ipynb so the rest of the
# code can be reused unchanged.

# clip_id    — unique label for each frame
# true_binary — 1 for fall, 0 for ADL for each overall video
feature_df['clip_id']     = feature_df['scene'] + '_' + feature_df['video']
feature_df['true_binary'] = (feature_df['true_label'] == 'FALL').astype(int)

print('unique clips:', feature_df['clip_id'].nunique())

print('\nlabel balance across all rows:')
print(feature_df['true_binary'].value_counts())

# Expect: 30 fall clips, 40 adl clips.
print('\nlabel balance across clips:')
print(feature_df.groupby('clip_id')['true_binary'].max().value_counts())


In [ ]:
feature_df['pose_detected'] = (
    feature_df['angle'].notna() & feature_df['hip_height'].notna()
)

# Per-clip detection rate — fraction of frames where we got a pose.
detection_rate = (
    feature_df.groupby('clip_id')['pose_detected']
              .mean()                          
              .reset_index()
              .rename(columns={'pose_detected': 'detection_rate'})
)

print('Per-clip detection rate stats:')
print(detection_rate['detection_rate'].describe())

print('\nClips with <50% detection (where we lose more frames than we keep):')
print(detection_rate[detection_rate['detection_rate'] < 0.5]
      .sort_values('detection_rate'))


In [ ]:
# This is the one cell where URFD's pipeline differs from Le2i's 
# as we need to account for the different FPS as it affects our hip velocity threshold.
#

FPS = 30   # URFD videos are 30 fps. Le2i was 25.

# diff() needs the rows in time order within each clip — otherwise we'd
# subtract frame 50 from frame 49 in some random order. sort first.
feature_df = (
    feature_df.sort_values(['clip_id', 'frame'])
              .reset_index(drop=True)
)

feature_df['hip_velocity'] = (
    feature_df.groupby('clip_id')['hip_height'].diff() * FPS
)


print('hip_velocity column added.')
print('\nlargest hip_velocity values seen:')
print(
    feature_df.dropna(subset=['hip_velocity'])
              .nlargest(5, 'hip_velocity')[
                  ['clip_id', 'frame', 'hip_height', 'hip_velocity']
              ]
)
print('\nIt is strange that the highest hip velocity were from ADL clips')
print('Also a hip velocity of near 6 seems unreasonable so will need to investigate')


In [ ]:
# State machine

# The three states:
#   state 0 (idle) - waiting for any downward velocity burst
#   state 1 (saw descent) - burst happened
#   state 2 (confirming) - posture seen at least once

def detect_fall_in_clip_or(group, V, N, P, max_gap, angle_thresh, bbox_thresh):
    # Sort by frame so the SM steps through time in order, not in DataFrame storage order
    g = group.sort_values('frame').reset_index(drop=True)

    state           = 0   # current SM State
    wait_counter    = 0   # frames spent in state 1
    persist_counter = 0   # frames posture has held in state 2
    gap_counter    = 0    # consecutive non-posture frames inside state 2

    for _, row in g.iterrows():
        vel   = row['hip_velocity']
        angle = row['angle']
        bbox  = row['bbox_ratio']

        posture = (
            (pd.notna(angle) and angle >= angle_thresh) or
            (pd.notna(bbox)  and bbox  >  bbox_thresh)
        )
        descent = pd.notna(vel) and vel > V

        if state == 0:
            if descent:
                state, wait_counter = 1, 0

        elif state == 1:
            wait_counter += 1
            if posture:
                state, persist_counter, gap_counter = 2, 1, 0
            elif wait_counter >= N:
                state = 0 

        elif state == 2:
            if posture:
                persist_counter += 1
                gap_counter = 0
                if persist_counter >= P:
                    return 1   # FALL
            else:
                gap_counter += 1
                if gap_counter > max_gap:
                    state, persist_counter, gap_counter = 0, 0, 0

    return 0


In [ ]:
FINAL_V       = 0.215   
FINAL_N       = 30      
FINAL_P       = 2       
FINAL_MAX_GAP = 12      
FINAL_ANGLE   = 45      
FINAL_BBOX    = 0.85    

# Ground truth at the clip level — one label per video.
clip_truth = (
    feature_df.groupby('clip_id')['true_binary']
              .max()               
              .reset_index()
              .rename(columns={'true_binary': 'true_clip'})
)

# Run the SM once per clip and collect (clip_id, prediction).
preds = []
for clip_id, group in feature_df.groupby('clip_id'):
    pred = detect_fall_in_clip_or(
        group,
        V           = FINAL_V,
        N           = FINAL_N,
        P           = FINAL_P,
        max_gap     = FINAL_MAX_GAP,
        angle_thresh= FINAL_ANGLE,
        bbox_thresh = FINAL_BBOX,
    )
    preds.append({'clip_id': clip_id, 'pred_clip': pred})

# Join predictions to ground truth. One row per clip, two columns we care
# about: pred_clip and true_clip
results = pd.DataFrame(preds).merge(clip_truth, on='clip_id')

print(f'Ran detector on {len(results)} clips.')
print(f'Predicted FALL: {(results["pred_clip"] == 1).sum()}')
print(f'Predicted ADL : {(results["pred_clip"] == 0).sum()}')
results.head()


In [ ]:
y_true = results['true_clip']
y_pred = results['pred_clip']

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()   

print('URFD held-out test — rule-based detector with Le2i-tuned thresholds')
print('              pred_nofall  pred_fall')
print(f'true_nofall   {tn:>11}  {fp:>9}')
print(f'true_fall     {fn:>11}  {tp:>9}')
print()
print(f'accuracy : {accuracy_score(y_true, y_pred):.3f}')
print(f'precision: {precision_score(y_true, y_pred, zero_division=0):.3f}')
print(f'recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}')
print(f'f1       : {f1_score(y_true, y_pred, zero_division=0):.3f}')


In [ ]:
# Pull the 23 ADL clips that the detector wrongly flagged as falls.
# These are the clips where true_clip == 0 (ADL) but pred_clip == 1 (fall).
fps = results[(results['true_clip'] == 0) & (results['pred_clip'] == 1)]
print(f'False positives: {len(fps)}')

# For each FP clip, pull the actual feature signals so we can see WHY it fired.
# The five columns we care about:

#   detection_rate  
#   max_angle     
#   max_bbox    
#   max_velocity    — anything > ~4 is almost certainly a MediaPipe tracking glitch, not real motion.
#   frames_either   — numbers of frames posture rules fire (angle or bbox)

fp_details = []
for cid in fps['clip_id']:
    clip = feature_df[feature_df['clip_id'] == cid]
    fp_details.append({
        'clip_id'        : cid,
        'detection_rate' : clip['pose_detected'].mean(),
        'max_angle'      : clip['angle'].max(),
        'max_bbox'       : clip['bbox_ratio'].max(),
        'max_velocity'   : clip['hip_velocity'].max(),
        'frames_angle_ok': (clip['angle']      >= FINAL_ANGLE).sum(),
        'frames_bbox_ok' : (clip['bbox_ratio'] >  FINAL_BBOX ).sum(),
        'frames_either'  : ((clip['angle']      >= FINAL_ANGLE) |
                            (clip['bbox_ratio'] >  FINAL_BBOX )).sum(),
    })

# Sort by max_velocity — clips at the top are likely glitch-driven FPs
fp_details_df = (
    pd.DataFrame(fp_details)
      .sort_values('max_velocity', ascending=False)
      .reset_index(drop=True)
)

print()
print(fp_details_df.to_string(index=False))


Going through each false positive and explaining why my state machine based approach failed. Trying to see 
if there is a justification for using LSTM


URFD_adl_adl-03 - LSTM should fix. squatting movement
URFD_adl_adl-04 - LSTM should fix. bending down 
URFD_adl_adl-38 - LSTM should fix. dropping to do a pushup
URFD_adl_adl-30 - LSTM should fix. lying down on floor
URFD_adl_adl-40 - LSTM should fix. lying down on floor
URFD_adl_adl-15 - LSTM should fix. bending over
URFD_adl_adl-33 - LSTM should fix. lying down
URFD_adl_adl-39 - LSTM should fix. lying down
URFD_adl_adl-23 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-17 - LSTM should fix. bending over
URFD_adl_adl-10 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-32 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-34 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-31 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-11 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-19 - LSTM should fix. sitting down quickly
URFD_adl_adl-22 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-35 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-37 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-36 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-05 - LSTM should fix. bending over
URFD_adl_adl-09 - LSTM should fix. sitting down
URFD_adl_adl-06 - LSTM should fix. bending over


Based on the false postives and the reason for them occuring, I believe an LSTM 
appoach will help solve a lot of incorrect predictions. Actions like sitting, 
sleeping, and bending over should be resolved, although actions like dropping to 
the floor to pick up an item may be quite hard for an LSTM to even catch.
    